## TabICL : principe

TabICL est un modèle préentraîné pour les données tabulaires. Nous utilisons **TabICLv2**, qui permet de faire de la régression pour prédire l’âge à partir des variables IRM.

Contrairement à un réseau entraîné de zéro, ses poids ont déjà été appris sur de nombreux jeux de données synthétiques. Il encode les colonnes, combine leurs informations pour représenter chaque personne, puis utilise les exemples dont l’âge est connu pour prédire les nouveaux âges : c’est l’**apprentissage en contexte** (*in-context learning*).

Dans son utilisation par défaut, `fit()` prépare et conserve les exemples d’entraînement, sans modifier les poids du réseau. Le modèle exploite ces exemples lors de `predict()`.

### Rappel : qu’est-ce qu’un Transformer ?

Un **Transformer** est une architecture de réseau de neurones fondée sur l’**attention** : chaque élément peut combiner les informations d’autres éléments en leur attribuant des poids selon leur pertinence. Ce mécanisme lui permet de modéliser des relations complexes et dépendantes du contexte.

TabICL partage ce principe avec TabPFN, mais son architecture et son préentraînement diffèrent. Il pourrait donc améliorer les prédictions ou apporter des erreurs complémentaires dans notre stacking.

In [1]:
%pip install -q -U tabicl

In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import r2_score
from tabicl import TabICLRegressor

SEED = 42

In [4]:
DATA_DIR = Path("/content")

# Données préparées
X_train = pd.read_csv(DATA_DIR / "X_train_log_knn.csv", index_col="id")
X_test = pd.read_csv(DATA_DIR / "X_test_log_knn.csv", index_col="id")

# Âges, alignés sur les personnes conservées
y_train = pd.read_csv(DATA_DIR / "y_train.csv", index_col="id")
y_train = y_train.loc[X_train.index, "y"]

# Folds communs, alignés sur les mêmes personnes
folds = pd.read_csv(DATA_DIR / "folds.csv", index_col="id")
folds = folds.loc[X_train.index, "fold"]

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")
print(f"Cible : {y_train.shape}")
print("\nNombre de personnes par fold :")
print(folds.value_counts().sort_index())

Train : (1152, 216)
Test  : (776, 216)
Cible : (1152,)

Nombre de personnes par fold :
fold
0    231
1    231
2    230
3    230
4    230
Name: count, dtype: int64


In [9]:
assert torch.cuda.is_available(), "Active le GPU dans les paramètres Colab."

DEVICE = "cuda"

def make_model():
    return TabICLRegressor(
        n_estimators=32,
        batch_size=1,
        device=DEVICE,
        random_state=SEED,
    )

def synchronize_device():
    torch.cuda.synchronize()

print(f"GPU utilisé : {torch.cuda.get_device_name(0)}")

GPU utilisé : Tesla T4


In [10]:
# Vérification de l’alignement
assert X_train.index.equals(y_train.index)
assert X_train.index.equals(folds.index)
assert set(folds.unique()) == set(range(5))

oof_tabicl = pd.Series(np.nan, index=X_train.index, name="y")
results = []

total_start = time.perf_counter()

for fold in range(5):
    train_mask = folds != fold
    valid_mask = folds == fold

    model = make_model()

    synchronize_device()
    start = time.perf_counter()

    model.fit(
        X_train.loc[train_mask],
        y_train.loc[train_mask],
    )
    predictions = model.predict(X_train.loc[valid_mask])

    synchronize_device()
    elapsed = time.perf_counter() - start

    oof_tabicl.loc[valid_mask] = predictions
    score = r2_score(y_train.loc[valid_mask], predictions)

    results.append({
        "fold": fold,
        "r2": score,
        "duration_s": elapsed,
    })

    print(
        f"Fold {fold} — R² : {score:.4f} — {elapsed:.1f} s",
        flush=True,
    )

    # Libérer le modèle avant le fold suivant
    del model
    torch.cuda.empty_cache()

assert np.isfinite(oof_tabicl.to_numpy()).all()

results = pd.DataFrame(results)

print(f"\nR² moyen des folds : {results['r2'].mean():.4f}")
print(f"R² OOF global      : {r2_score(y_train, oof_tabicl):.4f}")
print(f"Durée totale       : {time.perf_counter() - total_start:.1f} s")

oof_tabicl.to_csv(DATA_DIR / "oof_tabicl.csv", index_label="id")
results.to_csv(DATA_DIR / "cv_results_tabicl.csv", index=False)

print("\nPrédictions OOF et scores sauvegardés.")

Fold 0 — R² : 0.7939 — 10.1 s
Fold 1 — R² : 0.7894 — 9.9 s
Fold 2 — R² : 0.7066 — 9.9 s
Fold 3 — R² : 0.7678 — 10.0 s
Fold 4 — R² : 0.7712 — 9.8 s

R² moyen des folds : 0.7658
R² OOF global      : 0.7668
Durée totale       : 49.8 s

Prédictions OOF et scores sauvegardés.


In [11]:
from sklearn.linear_model import LinearRegression

# Prédictions OOF des trois modèles, alignées par id
predictions = pd.DataFrame(index=X_train.index)

for name in ["tabpfn", "gp"]:
    predictions[name] = pd.read_csv(
        DATA_DIR / f"oof_{name}.csv",
        index_col="id",
    ).iloc[:, 0].loc[X_train.index]

predictions["tabicl"] = oof_tabicl

assert np.isfinite(predictions.to_numpy()).all()

# Comparaison des deux stacks sur les mêmes folds
stacks = {
    "TabPFN + GP": ["tabpfn", "gp"],
    "TabPFN + GP + TabICL": ["tabpfn", "gp", "tabicl"],
}

stack_oof = pd.DataFrame(index=X_train.index)
scores = pd.DataFrame(index=range(5))
scores.index.name = "fold"

for name, columns in stacks.items():
    stack_oof[name] = np.nan

    for fold in range(5):
        train_mask = folds != fold
        valid_mask = folds == fold

        meta_model = LinearRegression()
        meta_model.fit(
            predictions.loc[train_mask, columns],
            y_train.loc[train_mask],
        )

        pred = meta_model.predict(predictions.loc[valid_mask, columns])
        stack_oof.loc[valid_mask, name] = pred

        scores.loc[fold, name] = r2_score(
            y_train.loc[valid_mask], pred
        )

scores["Gain TabICL"] = (
    scores["TabPFN + GP + TabICL"] - scores["TabPFN + GP"]
)

print("R² par fold :")
print(scores.round(4).to_string())

print("\nR² moyen des folds :")
print(scores.mean().round(4).to_string())

print("\nR² OOF global :")
for name in stacks:
    print(f"{name} : {r2_score(y_train, stack_oof[name]):.4f}")

R² par fold :
      TabPFN + GP  TabPFN + GP + TabICL  Gain TabICL
fold                                                
0          0.8117                0.8114      -0.0003
1          0.8011                0.8013       0.0002
2          0.7283                0.7266      -0.0017
3          0.7701                0.7702       0.0001
4          0.7838                0.7842       0.0003

R² moyen des folds :
TabPFN + GP             0.7790
TabPFN + GP + TabICL    0.7787
Gain TabICL            -0.0003

R² OOF global :
TabPFN + GP : 0.7800
TabPFN + GP + TabICL : 0.7798


In [13]:
# Données brutes : mêmes 1 152 personnes, NaN conservés
X_raw = pd.read_csv(
    DATA_DIR / "X_train.csv",
    index_col="id",
).loc[X_train.index].copy()

# Retirer les quatre colonnes constantes
X_raw = X_raw.drop(columns=["x104", "x129", "x489", "x530"])

# Reprendre les mêmes transformations logarithmiques
LOG_COLS = [
    "x115", "x159", "x415", "x458", "x507", "x531",
    "x542", "x641", "x665", "x702", "x742",
]

X_raw[LOG_COLS] = np.log(X_raw[LOG_COLS])

# Première variante : les mêmes 216 variables, sans imputation kNN
X_nan_216 = X_raw.loc[:, X_train.columns].copy()

assert X_nan_216.index.equals(y_train.index)
assert not np.isinf(X_raw.to_numpy()).any()

print(f"Données disponibles : {X_raw.shape}")
print(f"Variante 216        : {X_nan_216.shape}")
print(f"Valeurs manquantes  : {X_nan_216.isna().mean().mean():.1%}")

Données disponibles : (1152, 828)
Variante 216        : (1152, 216)
Valeurs manquantes  : 7.6%


In [14]:
# Résultats séparés pour conserver nos essais précédents
nan_oof = pd.DataFrame(
    np.nan,
    index=X_train.index,
    columns=["tabicl_nan_216", "tabicl_nan_300"],
)
nan_records = []

for fold in range(5):
    train_mask = folds != fold
    valid_mask = folds == fold

    # Classement calculé uniquement avec les âges d'entraînement
    ranking = (
        X_raw.loc[train_mask]
        .corrwith(y_train.loc[train_mask], method="spearman")
        .abs()
        .dropna()
        .sort_values(ascending=False)
    )

    feature_sets = {
        "tabicl_nan_216": X_nan_216.columns,
        "tabicl_nan_300": ranking.index[:300],
    }

    for name, columns in feature_sets.items():
        model_nan = TabICLRegressor(
            n_estimators=32,
            batch_size=1,
            device=DEVICE,
            random_state=SEED,
        )

        synchronize_device()
        start = time.perf_counter()

        model_nan.fit(
            X_raw.loc[train_mask, columns],
            y_train.loc[train_mask],
        )
        pred = model_nan.predict(X_raw.loc[valid_mask, columns])

        synchronize_device()
        elapsed = time.perf_counter() - start

        nan_oof.loc[valid_mask, name] = pred
        score = r2_score(y_train.loc[valid_mask], pred)

        nan_records.append({
            "model": name,
            "fold": fold,
            "r2": score,
            "duration_s": elapsed,
        })

        print(
            f"{name} | fold {fold} | R² : {score:.4f} "
            f"| {elapsed:.1f} s",
            flush=True,
        )

        del model_nan
        torch.cuda.empty_cache()

assert np.isfinite(nan_oof.to_numpy()).all()

nan_results = pd.DataFrame(nan_records)

print("\nRésultats :")
for name in nan_oof.columns:
    mean_score = nan_results.loc[nan_results["model"] == name, "r2"].mean()
    global_score = r2_score(y_train, nan_oof[name])

    print(
        f"{name} — R² moyen : {mean_score:.4f} "
        f"— R² OOF global : {global_score:.4f}"
    )

    nan_oof[name].to_csv(
        DATA_DIR / f"oof_{name}_32.csv",
        index_label="id",
    )

nan_results.to_csv(DATA_DIR / "cv_tabicl_nan_32.csv", index=False)

tabicl_nan_216 | fold 0 | R² : 0.7068 | 9.8 s
tabicl_nan_300 | fold 0 | R² : 0.7069 | 13.7 s
tabicl_nan_216 | fold 1 | R² : 0.7082 | 9.8 s
tabicl_nan_300 | fold 1 | R² : 0.6993 | 13.8 s
tabicl_nan_216 | fold 2 | R² : 0.6945 | 9.9 s
tabicl_nan_300 | fold 2 | R² : 0.6897 | 13.0 s
tabicl_nan_216 | fold 3 | R² : 0.6954 | 10.0 s
tabicl_nan_300 | fold 3 | R² : 0.6855 | 13.0 s
tabicl_nan_216 | fold 4 | R² : 0.7006 | 9.8 s
tabicl_nan_300 | fold 4 | R² : 0.7053 | 13.0 s

Résultats :
tabicl_nan_216 — R² moyen : 0.7011 — R² OOF global : 0.7014
tabicl_nan_300 — R² moyen : 0.6973 — R² OOF global : 0.6978


In [15]:
def score_stack(columns):
    pred = pd.Series(np.nan, index=y_train.index)

    for f in range(5):
        train = folds != f
        valid = folds == f

        meta = LinearRegression().fit(
            columns.loc[train], y_train.loc[train]
        )
        pred.loc[valid] = meta.predict(columns.loc[valid])

    return r2_score(y_train, pred)


# Référence précédemment chargée : TabPFN + GP
base = predictions.loc[y_train.index, ["tabpfn", "gp"]].copy()
reference_score = score_stack(base)

print(f"Référence : {reference_score:.5f}")

for name in nan_oof.columns:
    score = score_stack(base.assign(**{name: nan_oof[name]}))
    print(
        f"+ {name} : {score:.5f} "
        f"— gain : {score - reference_score:+.5f}"
    )

Référence : 0.78002
+ tabicl_nan_216 : 0.78101 — gain : +0.00100
+ tabicl_nan_300 : 0.77994 — gain : -0.00008


In [17]:
tuned_oof = pd.read_csv(
    DATA_DIR / "oof_tabpfn_nan_300_tuned.csv",
    index_col="id",
).iloc[:, 0].loc[y_train.index]

best_base = pd.DataFrame({
    "tabpfn_tuned": tuned_oof,
    "gp": base["gp"],
}).loc[y_train.index]

assert np.isfinite(best_base.to_numpy()).all()

reference = score_stack(best_base)
candidate = score_stack(
    best_base.assign(tabicl_nan_216=nan_oof["tabicl_nan_216"])
)

print(f"TabPFN optimisé + GP          : {reference:.5f}")
print(f"TabPFN optimisé + GP + TabICL : {candidate:.5f}")
print(f"Gain                         : {candidate - reference:+.5f}")

TabPFN optimisé + GP          : 0.78374
TabPFN optimisé + GP + TabICL : 0.78566
Gain                         : +0.00192


In [18]:
stack_inputs = {
    "Référence": best_base,
    "Avec TabICL": best_base.assign(
        tabicl_nan_216=nan_oof["tabicl_nan_216"]
    ),
}

comparison_oof = pd.DataFrame(
    np.nan,
    index=y_train.index,
    columns=stack_inputs.keys(),
)
fold_records = []

for f in range(5):
    train = folds != f
    valid = folds == f
    record = {"fold": f}

    for name, columns in stack_inputs.items():
        meta = LinearRegression().fit(
            columns.loc[train],
            y_train.loc[train],
        )

        pred = meta.predict(columns.loc[valid])
        comparison_oof.loc[valid, name] = pred
        record[name] = r2_score(y_train.loc[valid], pred)

    record["Gain"] = record["Avec TabICL"] - record["Référence"]
    fold_records.append(record)

fold_comparison = pd.DataFrame(fold_records).set_index("fold")

print(fold_comparison.round(5).to_string())
print(
    f"\nFolds améliorés : "
    f"{(fold_comparison['Gain'] > 0).sum()}/5"
)

      Référence  Avec TabICL     Gain
fold                                 
0       0.81910      0.82376  0.00466
1       0.80056      0.80530  0.00474
2       0.74847      0.74750 -0.00097
3       0.76618      0.76557 -0.00061
4       0.78051      0.78211  0.00160

Folds améliorés : 3/5


In [19]:
# Conserver les prédictions à 32 membres déjà calculées
factor_oof = {
    "knn_32": oof_tabicl.loc[y_train.index].copy(),
    "nan_32": nan_oof["tabicl_nan_216"].loc[y_train.index].copy(),
}

# Calculer uniquement les deux configurations manquantes
for preparation, X in {"knn": X_train, "nan": X_nan_216}.items():
    name = f"{preparation}_8"
    oof = pd.Series(np.nan, index=y_train.index, name="y")

    for f in range(5):
        train = folds != f
        valid = folds == f

        model_compare = TabICLRegressor(
            n_estimators=8,
            batch_size=1,
            norm_methods=None,  # Normalisation par défaut
            device=DEVICE,
            random_state=SEED,
        )

        model_compare.fit(X.loc[train], y_train.loc[train])
        oof.loc[valid] = model_compare.predict(X.loc[valid])
        synchronize_device()

        print(f"{name} — fold {f} terminé", flush=True)

        del model_compare
        torch.cuda.empty_cache()

    factor_oof[name] = oof


# Même évaluation du stack pour les quatre configurations
def predict_stack_oof(columns):
    pred = pd.Series(np.nan, index=y_train.index)

    for f in range(5):
        train = folds != f
        valid = folds == f

        meta = LinearRegression().fit(
            columns.loc[train], y_train.loc[train]
        )
        pred.loc[valid] = meta.predict(columns.loc[valid])

    return pred


reference_pred = predict_stack_oof(best_base)
reference_score = r2_score(y_train, reference_pred)

reference_folds = pd.Series({
    f: r2_score(y_train.loc[folds == f], reference_pred.loc[folds == f])
    for f in range(5)
})

factor_records = []
factor_gains = pd.DataFrame(index=range(5))
factor_gains.index.name = "fold"

for name, oof in factor_oof.items():
    assert np.isfinite(oof.to_numpy()).all()

    stack_pred = predict_stack_oof(best_base.assign(tabicl=oof))
    stack_score = r2_score(y_train, stack_pred)

    for f in range(5):
        valid = folds == f
        factor_gains.loc[f, name] = (
            r2_score(y_train.loc[valid], stack_pred.loc[valid])
            - reference_folds.loc[f]
        )

    factor_records.append({
        "configuration": name,
        "R2_TabICL": r2_score(y_train, oof),
        "R2_stack": stack_score,
        "gain_vs_reference": stack_score - reference_score,
        "folds_améliores": int((factor_gains[name] > 0).sum()),
    })

    oof.rename("y").to_csv(
        DATA_DIR / f"oof_tabicl_{name}.csv",
        index_label="id",
    )

factor_results = (
    pd.DataFrame(factor_records)
    .set_index("configuration")
    .sort_values("R2_stack", ascending=False)
)

factor_results.to_csv(DATA_DIR / "comparison_tabicl_knn_nan_8_32.csv")

print(f"\nRéférence TabPFN optimisé + GP : {reference_score:.5f}")
display(factor_results.round(5))

print("\nGains du stack par fold :")
display(factor_gains.round(5))

knn_8 — fold 0 terminé
knn_8 — fold 1 terminé
knn_8 — fold 2 terminé
knn_8 — fold 3 terminé
knn_8 — fold 4 terminé
nan_8 — fold 0 terminé
nan_8 — fold 1 terminé
nan_8 — fold 2 terminé
nan_8 — fold 3 terminé
nan_8 — fold 4 terminé

Référence TabPFN optimisé + GP : 0.78374


,R2_TabICL,R2_stack,gain_vs_reference,folds_améliores
configuration,,,,
nan_32,0.70135,0.78566,0.00192,3
nan_8,0.69764,0.78537,0.00163,3
knn_8,0.76285,0.78271,-0.00103,2
knn_32,0.76679,0.78247,-0.00127,3



Gains du stack par fold :


,knn_32,nan_32,knn_8,nan_8
fold,,,,
0,-0.00151,0.00466,-0.00160,0.00424
1,0.00082,0.00474,0.00029,0.00456
2,-0.00728,-0.00097,-0.00448,-0.00118
3,0.00033,-0.00061,-0.00002,-0.00122
4,0.00083,0.00160,0.00043,0.00155


In [20]:
# Réutiliser notre référence NaN à 8 membres
norm8_oof = {"default": factor_oof["nan_8"].copy()}

default8_score = score_stack(
    best_base.assign(tabicl=factor_oof["nan_8"])
)
default32_score = score_stack(
    best_base.assign(tabicl=factor_oof["nan_32"])
)

for method in ["none", "power", "quantile", "robust"]:
    oof = pd.Series(np.nan, index=y_train.index, name="y")
    start = time.perf_counter()

    for f in range(5):
        train = folds != f
        valid = folds == f

        model_norm = TabICLRegressor(
            n_estimators=8,
            batch_size=1,
            norm_methods=[method],
            device=DEVICE,
            random_state=SEED,
        )

        model_norm.fit(
            X_nan_216.loc[train],
            y_train.loc[train],
        )
        oof.loc[valid] = model_norm.predict(X_nan_216.loc[valid])
        synchronize_device()

        print(f"{method} — fold {f} terminé", flush=True)

        del model_norm
        torch.cuda.empty_cache()

    assert np.isfinite(oof.to_numpy()).all()
    norm8_oof[method] = oof

    score = score_stack(best_base.assign(tabicl=oof))
    print(
        f"→ Stack : {score:.5f} "
        f"| {time.perf_counter() - start:.0f} s\n",
        flush=True,
    )

# Comparer aux références à 8 et à 32 membres
records = []

for name, oof in norm8_oof.items():
    score = score_stack(best_base.assign(tabicl=oof))

    records.append({
        "normalisation": name,
        "R2_TabICL": r2_score(y_train, oof),
        "R2_stack": score,
        "gain_vs_default8": score - default8_score,
        "gain_vs_default32": score - default32_score,
    })

    oof.rename("y").to_csv(
        DATA_DIR / f"oof_tabicl_nan216_norm_{name}_8.csv",
        index_label="id",
    )

norm8_results = (
    pd.DataFrame(records)
    .set_index("normalisation")
    .sort_values("R2_stack", ascending=False)
)

norm8_results.to_csv(DATA_DIR / "tuning_tabicl_normalization_8.csv")
display(norm8_results.round(5))

none — fold 0 terminé
none — fold 1 terminé
none — fold 2 terminé
none — fold 3 terminé
none — fold 4 terminé
→ Stack : 0.78535 | 14 s

power — fold 0 terminé
power — fold 1 terminé
power — fold 2 terminé
power — fold 3 terminé
power — fold 4 terminé
→ Stack : 0.78532 | 17 s



/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (921). n_quantiles is set to n_samples.
  warnings.warn(


quantile — fold 0 terminé


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (921). n_quantiles is set to n_samples.
  warnings.warn(


quantile — fold 1 terminé


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (922). n_quantiles is set to n_samples.
  warnings.warn(


... (2 outputs left out)

quantile — fold 3 terminé


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_data.py:2829: UserWarning: n_quantiles (1000) is greater than the total number of samples (922). n_quantiles is set to n_samples.
  warnings.warn(


quantile — fold 4 terminé
→ Stack : 0.78491 | 17 s

robust — fold 0 terminé
robust — fold 1 terminé
robust — fold 2 terminé
robust — fold 3 terminé
robust — fold 4 terminé
→ Stack : 0.78558 | 14 s



,R2_TabICL,R2_stack,gain_vs_default8,gain_vs_default32
normalisation,,,,
robust,0.70304,0.78558,0.00021,-0.00008
default,0.69764,0.78537,0.00000,-0.00029
none,0.70210,0.78535,-0.00003,-0.00032
power,0.69866,0.78532,-0.00005,-0.00034
quantile,0.70692,0.78491,-0.00046,-0.00075


In [21]:
robust32_oof = pd.Series(np.nan, index=y_train.index, name="y")

for f in range(5):
    train = folds != f
    valid = folds == f

    model_robust = TabICLRegressor(
        n_estimators=32,
        batch_size=1,
        norm_methods=["robust"],
        device=DEVICE,
        random_state=SEED,
    )

    model_robust.fit(
        X_nan_216.loc[train],
        y_train.loc[train],
    )
    robust32_oof.loc[valid] = model_robust.predict(
        X_nan_216.loc[valid]
    )
    synchronize_device()

    print(f"robust, 32 membres — fold {f} terminé", flush=True)

    del model_robust
    torch.cuda.empty_cache()

assert np.isfinite(robust32_oof.to_numpy()).all()

reference = score_stack(
    best_base.assign(tabicl=factor_oof["nan_32"])
)
candidate = score_stack(
    best_base.assign(tabicl=robust32_oof)
)

print(f"\nNormalisation par défaut, 32 membres : {reference:.5f}")
print(f"Normalisation robust, 32 membres     : {candidate:.5f}")
print(f"Gain                                : {candidate - reference:+.5f}")

robust32_oof.to_csv(
    DATA_DIR / "oof_tabicl_nan216_norm_robust_32.csv",
    index_label="id",
)

robust, 32 membres — fold 0 terminé
robust, 32 membres — fold 1 terminé
robust, 32 membres — fold 2 terminé
robust, 32 membres — fold 3 terminé
robust, 32 membres — fold 4 terminé

Normalisation par défaut, 32 membres : 0.78566
Normalisation robust, 32 membres     : 0.78565
Gain                                : -0.00001


In [22]:
# Même préparation que notre variante NaN à 216 variables
raw_test = pd.read_csv(DATA_DIR / "X_test.csv", index_col="id")

X_test_nan_216 = raw_test.loc[
    X_test.index, X_nan_216.columns
].copy()

X_test_nan_216[LOG_COLS] = np.log(X_test_nan_216[LOG_COLS])

assert X_test_nan_216.columns.equals(X_nan_216.columns)
assert not np.isinf(X_test_nan_216.to_numpy()).any()

# Configuration retenue
final_tabicl = TabICLRegressor(
    n_estimators=32,
    batch_size=1,
    norm_methods=None,
    device=DEVICE,
    random_state=SEED,
)

# Toutes les personnes d'entraînement servent de contexte
final_tabicl.fit(X_nan_216, y_train)

tabicl_test = pd.Series(
    final_tabicl.predict(X_test_nan_216),
    index=X_test_nan_216.index,
    name="y",
)

assert np.isfinite(tabicl_test.to_numpy()).all()

tabicl_test.to_csv(
    DATA_DIR / "submission_tabicl_nan216_32.csv",
    index_label="id",
)

print(f"{len(tabicl_test)} prédictions test sauvegardées.")
display(tabicl_test.head())

776 prédictions test sauvegardées.


,y
id,
0.0,61.938194
1.0,76.731125
2.0,69.893394
3.0,75.703766
4.0,72.367363


In [23]:
from zipfile import ZipFile
from google.colab import files

archive = DATA_DIR / "tabicl_results.zip"

with ZipFile(archive, "w") as z:
    for name in [
        "oof_tabicl_nan_32.csv",
        "submission_tabicl_nan216_32.csv",
        "comparison_tabicl_knn_nan_8_32.csv",
    ]:
        z.write(DATA_DIR / name, arcname=name)

files.download(str(archive))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
%pip install -q "tabpfn==9.2.0"

In [26]:
import os
from getpass import getpass

os.environ["TABPFN_TOKEN"] = getpass("Clé API Prior Labs : ").strip()

from tabpfn import TabPFNRegressor

print("TabPFN importé.")

Clé API Prior Labs : ··········
TabPFN importé.


In [29]:
from tabpfn.browser_auth import get_cached_token, verify_token
from tabpfn.settings import settings

auth_token = get_cached_token()

if not auth_token:
    print("Aucune clé détectée dans cette session.")
else:
    status = verify_token(auth_token, settings.tabpfn.auth_api_url)
    print({
        True: "Clé reconnue et valide.",
        False: "Clé refusée ou expirée.",
        None: "Service d’authentification injoignable.",
    }[status])

Clé refusée ou expirée.


In [30]:
import os
from getpass import getpass
from tabpfn.browser_auth import verify_token
from tabpfn.settings import settings

new_key = getpass("API Key Prior Labs : ").strip()
status = verify_token(new_key, settings.tabpfn.auth_api_url)

if status is True:
    os.environ["TABPFN_TOKEN"] = new_key
    print("Clé valide et chargée dans Colab.")
elif status is False:
    print("Cette clé est également refusée.")
else:
    print("Service d’authentification injoignable.")

API Key Prior Labs : ··········
Clé valide et chargée dans Colab.


In [31]:
import gc

FEATURE_COUNTS = [150, 216, 300, 400]

fs_oof = pd.DataFrame(
    np.nan,
    index=y_train.index,
    columns=FEATURE_COUNTS,
)

for f in range(5):
    train = folds != f
    valid = folds == f

    # Sélection sans utiliser les âges du fold de validation
    ranking = (
        X_raw.loc[train]
        .corrwith(y_train.loc[train], method="spearman")
        .abs()
        .dropna()
        .sort_values(ascending=False)
    )

    assert len(ranking) >= max(FEATURE_COUNTS)

    for k in FEATURE_COUNTS:
        columns = ranking.index[:k]
        start = time.perf_counter()

        model_fs = TabPFNRegressor(
            n_estimators=8,
            softmax_temperature=1.0,
            average_before_softmax=True,
            random_state=0,
            device=DEVICE,
        )

        model_fs.fit(
            X_raw.loc[train, columns],
            y_train.loc[train],
        )
        fs_oof.loc[valid, k] = model_fs.predict(
            X_raw.loc[valid, columns]
        )
        synchronize_device()

        print(
            f"Fold {f} — {k} variables "
            f"— {time.perf_counter() - start:.1f} s",
            flush=True,
        )

        del model_fs
        gc.collect()
        torch.cuda.empty_cache()

assert np.isfinite(fs_oof.to_numpy()).all()

for k in FEATURE_COUNTS:
    fs_oof[k].rename("y").to_csv(
        DATA_DIR / f"oof_tabpfn_fs_{k}.csv",
        index_label="id",
    )

print("\nLes quatre variantes sont calculées et sauvegardées.")

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

Fold 0 — 150 variables — 17.7 s
Fold 0 — 216 variables — 5.2 s
Fold 0 — 300 variables — 6.3 s
Fold 0 — 400 variables — 8.6 s
Fold 1 — 150 variables — 3.3 s
Fold 1 — 216 variables — 4.9 s
Fold 1 — 300 variables — 6.2 s
Fold 1 — 400 variables — 8.1 s
Fold 2 — 150 variables — 3.2 s
Fold 2 — 216 variables — 4.8 s
Fold 2 — 300 variables — 6.0 s
Fold 2 — 400 variables — 8.0 s
Fold 3 — 150 variables — 3.2 s
Fold 3 — 216 variables — 4.7 s
Fold 3 — 300 variables — 6.2 s
Fold 3 — 400 variables — 8.1 s
Fold 4 — 150 variables — 3.2 s
Fold 4 — 216 variables — 4.8 s
Fold 4 — 300 variables — 6.3 s
Fold 4 — 400 variables — 8.1 s

Les quatre variantes sont calculées et sauvegardées.


In [32]:
fs_stack_oof = pd.DataFrame(index=y_train.index)

for k in FEATURE_COUNTS:
    inputs = pd.DataFrame({
        "tabpfn": fs_oof[k],
        "gp": best_base["gp"],
        "tabicl": factor_oof["nan_32"],
    })

    fs_stack_oof[k] = predict_stack_oof(inputs)

# Scores globaux
fs_results = pd.DataFrame(
    {
        k: {
            "R2_TabPFN": r2_score(y_train, fs_oof[k]),
            "R2_stack": r2_score(y_train, fs_stack_oof[k]),
        }
        for k in FEATURE_COUNTS
    }
).T

fs_results.index.name = "n_features"
fs_results["gain_vs_300"] = (
    fs_results["R2_stack"] - fs_results.loc[300, "R2_stack"]
)

# Gains par fold par rapport à la variante à 300 variables
fs_fold_gains = pd.DataFrame(index=range(5))
fs_fold_gains.index.name = "fold"

for f in range(5):
    valid = folds == f
    reference = r2_score(
        y_train.loc[valid],
        fs_stack_oof.loc[valid, 300],
    )

    for k in FEATURE_COUNTS:
        fs_fold_gains.loc[f, k] = (
            r2_score(y_train.loc[valid], fs_stack_oof.loc[valid, k])
            - reference
        )

# Sauvegarde
fs_results.to_csv(DATA_DIR / "tabpfn_feature_counts_results.csv")
fs_fold_gains.to_csv(DATA_DIR / "tabpfn_feature_counts_fold_gains.csv")

display(fs_results.sort_values("R2_stack", ascending=False).round(5))

print("Gains par fold par rapport à 300 variables :")
display(fs_fold_gains.round(5))

,R2_TabPFN,R2_stack,gain_vs_300
n_features,,,
216,0.77582,0.78076,0.00404
300,0.77094,0.77672,0.00000
400,0.77036,0.77640,-0.00032
150,0.76902,0.77521,-0.00152


Gains par fold par rapport à 300 variables :


,150,216,300,400
fold,,,,
0,0.00646,0.00986,0.0,0.00268
1,-0.00279,-0.00177,0.0,0.00027
2,-0.00512,0.00468,0.0,0.00267
3,-0.00019,-0.00044,0.0,-0.00869
4,-0.00620,0.00703,0.0,0.00125


In [33]:
comparison_rows = []

for k in FEATURE_COUNTS:
    inputs = pd.DataFrame({
        "tabpfn": fs_oof[k],
        "gp": best_base["gp"],
    })

    without_tabicl = score_stack(inputs)
    with_tabicl = score_stack(
        inputs.assign(tabicl=factor_oof["nan_32"])
    )

    comparison_rows.append({
        "n_features": k,
        "TabPFN + GP": without_tabicl,
        "TabPFN + GP + TabICL": with_tabicl,
        "gain_TabICL": with_tabicl - without_tabicl,
    })

fs_combinations = (
    pd.DataFrame(comparison_rows)
    .set_index("n_features")
)

display(fs_combinations.round(5))

,TabPFN + GP,TabPFN + GP + TabICL,gain_TabICL
n_features,,,
150,0.77540,0.77521,-0.00019
216,0.77962,0.78076,0.00114
300,0.77639,0.77672,0.00033
400,0.77586,0.77640,0.00055


In [34]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer

et_selection_oof = pd.Series(
    np.nan, index=y_train.index, name="y"
)
et_selected_columns = {}

for f in range(5):
    train = folds != f
    valid = folds == f

    # La médiane sert uniquement au classement par Extra Trees.
    # TabPFN recevra toujours les données avec NaN.
    imputer = SimpleImputer(
        strategy="median",
        keep_empty_features=True,
    )
    X_selection = imputer.fit_transform(X_raw.loc[train])

    selector = ExtraTreesRegressor(
        n_estimators=300,  # Arbres utilisés pour classer les variables
        min_samples_leaf=5,
        max_features=0.7,
        random_state=42,
        n_jobs=-1,
    )
    selector.fit(X_selection, y_train.loc[train])

    importance = pd.Series(
        selector.feature_importances_,
        index=X_raw.columns,
    )
    columns = importance.nlargest(216).index
    et_selected_columns[f] = columns.tolist()

    model_et = TabPFNRegressor(
        n_estimators=8,  # Même configuration TabPFN que précédemment
        softmax_temperature=1.0,
        average_before_softmax=True,
        random_state=0,
        device=DEVICE,
    )

    model_et.fit(
        X_raw.loc[train, columns],
        y_train.loc[train],
    )
    et_selection_oof.loc[valid] = model_et.predict(
        X_raw.loc[valid, columns]
    )
    synchronize_device()

    print(f"Sélection Extra Trees — fold {f} terminé", flush=True)

    del model_et, selector, X_selection
    gc.collect()
    torch.cuda.empty_cache()

assert np.isfinite(et_selection_oof.to_numpy()).all()

# Comparaison avec notre sélection Spearman à 216 variables
records = []

for name, oof in {
    "Spearman": fs_oof[216],
    "Extra Trees": et_selection_oof,
}.items():
    inputs = pd.DataFrame({
        "tabpfn": oof,
        "gp": best_base["gp"],
    })

    records.append({
        "selection": name,
        "R2_TabPFN": r2_score(y_train, oof),
        "TabPFN + GP": score_stack(inputs),
        "TabPFN + GP + TabICL": score_stack(
            inputs.assign(tabicl=factor_oof["nan_32"])
        ),
    })

selection_results = pd.DataFrame(records).set_index("selection")

# Sauvegarde
et_selection_oof.to_csv(
    DATA_DIR / "oof_tabpfn_selection_extratrees_216.csv",
    index_label="id",
)
pd.DataFrame(et_selected_columns).to_csv(
    DATA_DIR / "tabpfn_extratrees_features_by_fold.csv",
    index=False,
)
selection_results.to_csv(DATA_DIR / "tabpfn_selection_comparison.csv")

display(selection_results.round(5))

Sélection Extra Trees — fold 0 terminé
Sélection Extra Trees — fold 1 terminé
Sélection Extra Trees — fold 2 terminé
Sélection Extra Trees — fold 3 terminé
Sélection Extra Trees — fold 4 terminé


,R2_TabPFN,TabPFN + GP,TabPFN + GP + TabICL
selection,,,
Spearman,0.77582,0.77962,0.78076
Extra Trees,0.77477,0.77953,0.78023


In [35]:
# Moyenne des deux variantes TabPFN
blend_oof = (fs_oof[216] + et_selection_oof) / 2

blend_inputs = pd.DataFrame({
    "tabpfn": blend_oof,
    "gp": best_base["gp"],
})

blend_results = selection_results.copy()

blend_results.loc["Moyenne 50/50", "R2_TabPFN"] = r2_score(
    y_train, blend_oof
)
blend_results.loc["Moyenne 50/50", "TabPFN + GP"] = score_stack(
    blend_inputs
)
blend_results.loc["Moyenne 50/50", "TabPFN + GP + TabICL"] = score_stack(
    blend_inputs.assign(tabicl=factor_oof["nan_32"])
)

blend_oof.rename("y").to_csv(
    DATA_DIR / "oof_tabpfn_blend_spearman_extratrees.csv",
    index_label="id",
)

display(blend_results.round(5))

,R2_TabPFN,TabPFN + GP,TabPFN + GP + TabICL
selection,,,
Spearman,0.77582,0.77962,0.78076
Extra Trees,0.77477,0.77953,0.78023
Moyenne 50/50,0.77755,0.78129,0.78289


In [36]:
# Toutes les personnes, avec les mêmes colonnes que X_raw
X_all = pd.read_csv(
    DATA_DIR / "X_train.csv",
    index_col="id",
).loc[:, X_raw.columns].copy()

y_all = pd.read_csv(
    DATA_DIR / "y_train.csv",
    index_col="id",
)["y"].loc[X_all.index]

extra_ids = X_all.index.difference(y_train.index, sort=False)

# Repérer les valeurs incompatibles avec le logarithme
invalid_logs = X_all[LOG_COLS].le(0)
affected_rows = invalid_logs.loc[extra_ids].any(axis=1).sum()

# Logarithme des valeurs positives ; les autres deviennent NaN
X_all[LOG_COLS] = np.log(
    X_all[LOG_COLS].where(X_all[LOG_COLS] > 0)
)

# Vérifier que les 1 152 personnes déjà utilisées restent identiques
pd.testing.assert_frame_equal(
    X_all.loc[X_raw.index],
    X_raw,
)

assert np.isfinite(y_all.to_numpy()).all()
assert not np.isinf(X_all.to_numpy()).any()

print(f"Personnes disponibles : {len(X_all)}")
print(f"Personnes à réintégrer : {len(extra_ids)}")
print(f"Parmi elles, lignes avec log invalide traité en NaN : {affected_rows}")

Personnes disponibles : 1212
Personnes à réintégrer : 60
Parmi elles, lignes avec log invalide traité en NaN : 38


In [37]:
extra_oof = pd.DataFrame(
    np.nan,
    index=y_train.index,
    columns=["spearman", "extra_trees"],
)

for f in range(5):
    valid_ids = y_train.index[folds == f]

    # Personnes habituelles d'entraînement + les 60 réintégrées
    train_ids = y_train.index[folds != f].append(extra_ids)

    assert train_ids.intersection(valid_ids).empty

    X_fit = X_all.loc[train_ids]
    y_fit = y_all.loc[train_ids]

    # Classement Spearman
    spearman_columns = (
        X_fit.corrwith(y_fit, method="spearman")
        .abs()
        .dropna()
        .nlargest(216)
        .index
    )

    # Classement Extra Trees, avec les mêmes réglages que précédemment
    imputer = SimpleImputer(
        strategy="median",
        keep_empty_features=True,
    )
    X_selection = imputer.fit_transform(X_fit)

    selector = ExtraTreesRegressor(
        n_estimators=300,
        min_samples_leaf=5,
        max_features=0.7,
        random_state=42,
        n_jobs=-1,
    ).fit(X_selection, y_fit)

    tree_columns = pd.Series(
        selector.feature_importances_,
        index=X_fit.columns,
    ).nlargest(216).index

    feature_sets = {
        "spearman": spearman_columns,
        "extra_trees": tree_columns,
    }

    del selector, X_selection

    for name, columns in feature_sets.items():
        model_extra = TabPFNRegressor(
            n_estimators=8,
            softmax_temperature=1.0,
            average_before_softmax=True,
            random_state=0,
            device=DEVICE,
        )

        model_extra.fit(X_fit.loc[:, columns], y_fit)

        extra_oof.loc[valid_ids, name] = model_extra.predict(
            X_all.loc[valid_ids, columns]
        )
        synchronize_device()

        print(f"Fold {f} — {name} terminé", flush=True)

        del model_extra
        gc.collect()
        torch.cuda.empty_cache()

assert np.isfinite(extra_oof.to_numpy()).all()

for name in extra_oof.columns:
    extra_oof[name].rename("y").to_csv(
        DATA_DIR / f"oof_tabpfn_{name}_with_extra60.csv",
        index_label="id",
    )

print("Prédictions calculées. Lance maintenant la deuxième cellule.")

Fold 0 — spearman terminé
Fold 0 — extra_trees terminé
Fold 1 — spearman terminé
Fold 1 — extra_trees terminé
Fold 2 — spearman terminé
Fold 2 — extra_trees terminé
Fold 3 — spearman terminé
Fold 3 — extra_trees terminé
Fold 4 — spearman terminé
Fold 4 — extra_trees terminé
Prédictions calculées. Lance maintenant la deuxième cellule.


In [38]:
extra_blend_oof = extra_oof.mean(axis=1)

sources = {
    "Sans les 60": blend_oof,
    "Avec les 60": extra_blend_oof,
}

extra_records = []
extra_stack_preds = pd.DataFrame(index=y_train.index)

for name, predictions in sources.items():
    inputs = pd.DataFrame({
        "tabpfn": predictions,
        "gp": best_base["gp"],
    })

    stack_pred = predict_stack_oof(
        inputs.assign(tabicl=factor_oof["nan_32"])
    )
    extra_stack_preds[name] = stack_pred

    extra_records.append({
        "configuration": name,
        "R2_TabPFN_moyenne": r2_score(y_train, predictions),
        "TabPFN + GP": score_stack(inputs),
        "TabPFN + GP + TabICL": r2_score(y_train, stack_pred),
    })

extra_results = pd.DataFrame(extra_records).set_index("configuration")

extra_fold_results = pd.DataFrame({
    name: [
        r2_score(
            y_train.loc[folds == f],
            extra_stack_preds.loc[folds == f, name],
        )
        for f in range(5)
    ]
    for name in sources
})
extra_fold_results.index.name = "fold"
extra_fold_results["Gain"] = (
    extra_fold_results["Avec les 60"] - extra_fold_results["Sans les 60"]
)

extra_results.to_csv(DATA_DIR / "tabpfn_extra60_results.csv")
extra_fold_results.to_csv(DATA_DIR / "tabpfn_extra60_fold_results.csv")

display(extra_results.round(5))
print("Comparaison du stack complet par fold :")
display(extra_fold_results.round(5))

,R2_TabPFN_moyenne,TabPFN + GP,TabPFN + GP + TabICL
configuration,,,
Sans les 60,0.77755,0.78129,0.78289
Avec les 60,0.77739,0.78107,0.78198


Comparaison du stack complet par fold :


,Sans les 60,Avec les 60,Gain
fold,,,
0,0.82211,0.82074,-0.00136
1,0.80162,0.80914,0.00752
2,0.73410,0.72343,-0.01068
3,0.76559,0.76640,0.00080
4,0.78567,0.78468,-0.00099


In [39]:
extra_combinations = {
    "Sans les 60": blend_oof,
    "+60 pour Spearman seulement": (
        extra_oof["spearman"] + et_selection_oof
    ) / 2,
    "+60 pour Extra Trees seulement": (
        fs_oof[216] + extra_oof["extra_trees"]
    ) / 2,
    "+60 pour les deux": extra_oof.mean(axis=1),
}

records = []

for name, predictions in extra_combinations.items():
    inputs = pd.DataFrame({
        "tabpfn": predictions,
        "gp": best_base["gp"],
    })

    records.append({
        "configuration": name,
        "TabPFN + GP": score_stack(inputs),
        "TabPFN + GP + TabICL": score_stack(
            inputs.assign(tabicl=factor_oof["nan_32"])
        ),
    })

extra_combination_results = (
    pd.DataFrame(records).set_index("configuration")
)

extra_combination_results["gain_vs_reference"] = (
    extra_combination_results["TabPFN + GP + TabICL"]
    - extra_combination_results.loc[
        "Sans les 60", "TabPFN + GP + TabICL"
    ]
)

extra_combination_results.to_csv(
    DATA_DIR / "tabpfn_extra60_combinations.csv"
)

display(extra_combination_results.round(5))

,TabPFN + GP,TabPFN + GP + TabICL,gain_vs_reference
configuration,,,
Sans les 60,0.78129,0.78289,0.00000
+60 pour Spearman seulement,0.78287,0.78476,0.00187
+60 pour Extra Trees seulement,0.78119,0.78277,-0.00012
+60 pour les deux,0.78107,0.78198,-0.00090


In [40]:
candidate_blend_oof = extra_combinations[
    "+60 pour Spearman seulement"
].copy()

stack_predictions = {}

for name, predictions in {
    "Référence": blend_oof,
    "Candidat": candidate_blend_oof,
}.items():
    inputs = pd.DataFrame({
        "tabpfn": predictions,
        "gp": best_base["gp"],
        "tabicl": factor_oof["nan_32"],
    })
    stack_predictions[name] = predict_stack_oof(inputs)

candidate_fold_results = pd.DataFrame({
    name: [
        r2_score(
            y_train.loc[folds == f],
            predictions.loc[folds == f],
        )
        for f in range(5)
    ]
    for name, predictions in stack_predictions.items()
})

candidate_fold_results.index.name = "fold"
candidate_fold_results["Gain"] = (
    candidate_fold_results["Candidat"]
    - candidate_fold_results["Référence"]
)

candidate_blend_oof.rename("y").to_csv(
    DATA_DIR / "oof_tabpfn_spearman1212_extratrees1152.csv",
    index_label="id",
)

display(candidate_fold_results.round(5))
print(
    f"Folds améliorés : "
    f"{(candidate_fold_results['Gain'] > 0).sum()}/5"
)

,Référence,Candidat,Gain
fold,,,
0,0.82211,0.82333,0.00122
1,0.80162,0.80902,0.00740
2,0.73410,0.72960,-0.00450
3,0.76559,0.76821,0.00262
4,0.78567,0.78815,0.00248


Folds améliorés : 4/5


In [43]:
sample = pd.read_csv(DATA_DIR / "sample.csv")
test_ids = pd.Index(sample["id"], name="id")

assert list(sample.columns) == ["id", "y"]
assert len(sample) == 776 and test_ids.is_unique

# Toutes les variables disponibles, avec la même préparation
X_test_final = pd.read_csv(
    DATA_DIR / "X_test.csv", index_col="id"
).loc[test_ids, X_raw.columns].copy()

X_test_final[LOG_COLS] = np.log(
    X_test_final[LOG_COLS].where(X_test_final[LOG_COLS] > 0)
)

# GP et TabICL : leurs prédictions finales existent déjà
gp_test = pd.read_csv(
    DATA_DIR / "submission_gp.csv", index_col="id"
)["y"]

icl_test = pd.read_csv(
    DATA_DIR / "submission_tabicl_nan216_32.csv", index_col="id"
)["y"]

for predictions in [gp_test, icl_test]:
    assert predictions.index.is_unique
    assert set(predictions.index) == set(test_ids)
    assert np.isfinite(predictions.to_numpy()).all()

print("Données test et prédictions GP/TabICL prêtes.")

Données test et prédictions GP/TabICL prêtes.


In [44]:
FINAL_TABPFN_PARAMS = {
    "n_estimators": 8,
    "softmax_temperature": 1.0,
    "average_before_softmax": True,
    "random_state": 0,
    "device": DEVICE,
}

FINAL_SELECTOR_PARAMS = {
    "n_estimators": 300,
    "min_samples_leaf": 5,
    "max_features": 0.7,
    "random_state": 42,
    "n_jobs": -1,
}

# Sélection Spearman sur les 1 212 personnes
final_spearman_columns = (
    X_all.corrwith(y_all, method="spearman")
    .abs()
    .dropna()
    .nlargest(216)
    .index
)

# Sélection Extra Trees sur les 1 152 personnes
imputer = SimpleImputer(strategy="median", keep_empty_features=True)

selector = ExtraTreesRegressor(**FINAL_SELECTOR_PARAMS).fit(
    imputer.fit_transform(X_raw), y_train
)

final_tree_columns = pd.Series(
    selector.feature_importances_,
    index=X_raw.columns,
).nlargest(216).index

test_components = pd.DataFrame(index=test_ids)

for name, X_fit, y_fit, columns in [
    ("spearman", X_all, y_all, final_spearman_columns),
    ("extra_trees", X_raw, y_train, final_tree_columns),
]:
    print(f"Calcul du TabPFN {name}...", flush=True)

    model_final = TabPFNRegressor(**FINAL_TABPFN_PARAMS)
    model_final.fit(X_fit.loc[:, columns], y_fit)

    test_components[name] = model_final.predict(
        X_test_final.loc[:, columns]
    )
    synchronize_device()

    del model_final
    gc.collect()
    torch.cuda.empty_cache()

test_components["tabpfn_blend"] = (
    test_components["spearman"] + test_components["extra_trees"]
) / 2

test_components["gp"] = gp_test.loc[test_ids]
test_components["tabicl"] = icl_test.loc[test_ids]

assert np.isfinite(test_components.to_numpy()).all()
print("Les prédictions finales des quatre modèles sont prêtes.")

Calcul du TabPFN spearman...
Calcul du TabPFN extra_trees...
Les prédictions finales des quatre modèles sont prêtes.


In [45]:
import json
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

# OOF correspondant exactement au candidat retenu
final_oof = pd.DataFrame({
    "tabpfn_blend": (extra_oof["spearman"] + et_selection_oof) / 2,
    "gp": best_base["gp"],
    "tabicl": factor_oof["nan_32"],
}).loc[y_train.index]

assert np.isfinite(final_oof.to_numpy()).all()

final_cv = predict_stack_oof(final_oof)
final_cv_score = r2_score(y_train, final_cv)

assert abs(final_cv_score - 0.78476) < 0.00005, (
    f"Les OOF ne reproduisent pas le candidat : {final_cv_score:.6f}"
)

# Apprendre les coefficients finaux sur toutes les OOF
final_meta = LinearRegression().fit(final_oof, y_train)

submission = sample.copy()
submission["y"] = final_meta.predict(
    test_components.loc[test_ids, final_oof.columns]
)

assert np.isfinite(submission["y"]).all()

# Dossier de sauvegarde dans Colab
export_dir = DATA_DIR / "kaggle_candidate"
export_dir.mkdir(exist_ok=True)

submission.to_csv(
    export_dir / "submission_stack_tabpfn_gp_tabicl.csv",
    index=False,
)
test_components.to_csv(export_dir / "test_predictions.csv", index_label="id")
final_oof.to_csv(export_dir / "oof_models.csv", index_label="id")

pd.DataFrame({
    "y_true": y_train,
    "fold": folds,
    "stack_prediction": final_cv,
}).to_csv(export_dir / "validation.csv", index_label="id")

configuration = {
    "cv_r2": float(final_cv_score),
    "training_rows": {"spearman": 1212, "extra_trees": 1152},
    "tabpfn_parameters": FINAL_TABPFN_PARAMS,
    "selector_parameters": FINAL_SELECTOR_PARAMS,
    "spearman_features": final_spearman_columns.tolist(),
    "extra_trees_features": final_tree_columns.tolist(),
    "log_columns": LOG_COLS,
    "blend_weights": [0.5, 0.5],
    "meta_columns": final_oof.columns.tolist(),
    "meta_coefficients": final_meta.coef_.tolist(),
    "meta_intercept": float(final_meta.intercept_),
    "tabicl": {"n_estimators": 32, "norm_methods": None, "random_state": 42},
}

(export_dir / "configuration.json").write_text(
    json.dumps(configuration, indent=2)
)

# Télécharger le dossier complet sur ton Mac
archive = DATA_DIR / "kaggle_candidate.zip"

with ZipFile(archive, "w", compression=ZIP_DEFLATED) as z:
    for path in sorted(export_dir.iterdir()):
        if path.is_file():
            z.write(path, arcname=path.name)

print(f"R² de validation vérifié : {final_cv_score:.5f}")
display(submission.head())
files.download(str(archive))

R² de validation vérifié : 0.78476


,id,y
0,0.0,61.331794
1,1.0,75.789804
2,2.0,71.521569
3,3.0,75.348525
4,4.0,73.725506


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>